# Task : End-to-End ML Pipeline – Customer Churn Prediction

## Problem Statement
Build a production-ready, reusable ML pipeline to predict whether a telecom customer will churn.

## Objective
- Implement preprocessing pipelines using `sklearn.pipeline.Pipeline`
- Train Logistic Regression and Random Forest models
- Hyperparameter tuning with `GridSearchCV`
- Export the complete pipeline using `joblib`

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)

np.random.seed(42)
print('All libraries loaded.')

## 1. Dataset Loading & Preprocessing

In [ ]:
# Load Telco Churn dataset (download from Kaggle or use URL)
# Option A – if file is available locally:
# df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Option B – download directly
import urllib.request
url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/Telecom-Customer-Churn.csv'
urllib.request.urlretrieve(url, 'telco_churn.csv')
df = pd.read_csv('telco_churn.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# ── Data Quality Check ───────────────────────────────────────────────────────
print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

print('\n=== Data Types ===')
print(df.dtypes.value_counts())

print('\n=== Target Distribution ===')
print(df['Churn'].value_counts())
print(f'Churn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')

In [ ]:
# ── EDA Visualisations ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Telco Churn – Exploratory Data Analysis', fontsize=14, fontweight='bold')

# Churn distribution
df['Churn'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#2ecc71','#e74c3c'], rot=0)
axes[0,0].set_title('Churn Distribution')
axes[0,0].set_ylabel('Count')

# Tenure vs Churn
df.boxplot(column='tenure', by='Churn', ax=axes[0,1])
axes[0,1].set_title('Tenure by Churn')

# Monthly charges
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[0,2])
axes[0,2].set_title('Monthly Charges by Churn')

# Contract type
ct = df.groupby(['Contract','Churn']).size().unstack()
ct.plot(kind='bar', ax=axes[1,0], rot=15, color=['#2ecc71','#e74c3c'])
axes[1,0].set_title('Churn by Contract Type')

# Internet service
isvc = df.groupby(['InternetService','Churn']).size().unstack()
isvc.plot(kind='bar', ax=axes[1,1], rot=15)
axes[1,1].set_title('Churn by Internet Service')

# Correlation heatmap (numeric cols)
num_cols = df.select_dtypes(include=np.number).columns
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[1,2], annot=True, fmt='.2f', cmap='coolwarm')
axes[1,2].set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig('task2_eda.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────

# Fix TotalCharges (may have spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop customerID – not predictive
df.drop(columns=['customerID'], inplace=True, errors='ignore')

# Encode target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Identify column types
TARGET = 'Churn'
NUMERIC_COLS = df.select_dtypes(include=['int64','float64']).columns.drop(TARGET).tolist()
CATEGORICAL_COLS = df.select_dtypes(include=['object']).columns.tolist()

print(f'Numeric features  : {NUMERIC_COLS}')
print(f'Categorical features: {CATEGORICAL_COLS}')

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')

## 2. Pipeline Construction

In [ ]:
# ── Preprocessing sub-pipelines ───────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, NUMERIC_COLS),
    ('cat', categorical_transformer, CATEGORICAL_COLS)
])

# ── Full Pipelines ────────────────────────────────────────────────────────────
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(random_state=42, n_jobs=-1))
])

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   GradientBoostingClassifier(random_state=42))
])

print('Pipelines constructed.')

## 3. Model Training & GridSearchCV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── Logistic Regression GridSearch ───────────────────────────────────────────
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__solver': ['lbfgs', 'liblinear']
}
lr_gs = GridSearchCV(lr_pipeline, lr_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
lr_gs.fit(X_train, y_train)
print(f'LR  – Best AUC: {lr_gs.best_score_:.4f} | Params: {lr_gs.best_params_}')

In [ ]:
# ── Random Forest GridSearch ──────────────────────────────────────────────────
rf_param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}
rf_gs = GridSearchCV(rf_pipeline, rf_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1)
rf_gs.fit(X_train, y_train)
print(f'RF  – Best AUC: {rf_gs.best_score_:.4f} | Params: {rf_gs.best_params_}')

## 4. Evaluation

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f'\n── {name} ──────────────────────────────────')
    print(f'  Accuracy : {accuracy_score(y_test, y_pred):.4f}')
    print(f'  F1 Score : {f1_score(y_test, y_pred):.4f}')
    print(f'  ROC AUC  : {roc_auc_score(y_test, y_proba):.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Churn','Churn']))
    return y_pred, y_proba

lr_pred, lr_proba = evaluate_model('Logistic Regression', lr_gs.best_estimator_, X_test, y_test)
rf_pred, rf_proba = evaluate_model('Random Forest',       rf_gs.best_estimator_, X_test, y_test)

In [ ]:
# ── Visual Evaluation ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model Evaluation – Churn Prediction', fontsize=14, fontweight='bold')

# Confusion matrices
for ax, name, preds in zip(
    [axes[0,0], axes[0,1]],
    ['Logistic Regression', 'Random Forest'],
    [lr_pred, rf_pred]
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn','Churn'], yticklabels=['No Churn','Churn'])
    ax.set_title(f'{name}\nConfusion Matrix')

# ROC curves
RocCurveDisplay.from_predictions(y_test, lr_proba, ax=axes[0,2], name='Logistic Regression')
RocCurveDisplay.from_predictions(y_test, rf_proba, ax=axes[0,2], name='Random Forest')
axes[0,2].set_title('ROC Curves')
axes[0,2].grid(alpha=0.3)

# Precision-Recall
PrecisionRecallDisplay.from_predictions(y_test, lr_proba, ax=axes[1,0], name='LR')
PrecisionRecallDisplay.from_predictions(y_test, rf_proba, ax=axes[1,0], name='RF')
axes[1,0].set_title('Precision-Recall Curve')

# Feature importances (RF)
best_rf   = rf_gs.best_estimator_
feat_names = (NUMERIC_COLS +
    best_rf.named_steps['preprocessor']
             .named_transformers_['cat']
             .named_steps['onehot']
             .get_feature_names_out(CATEGORICAL_COLS).tolist())
importances = best_rf.named_steps['classifier'].feature_importances_
top_n = 15
indices = np.argsort(importances)[-top_n:]
axes[1,1].barh([feat_names[i] for i in indices], importances[indices], color='steelblue')
axes[1,1].set_title(f'Top {top_n} Feature Importances (RF)')
axes[1,1].grid(alpha=0.3)

# GridSearch CV results
cv_results = pd.DataFrame(rf_gs.cv_results_)
axes[1,2].scatter(range(len(cv_results)), cv_results['mean_test_score'],
                  c=cv_results['mean_test_score'], cmap='viridis')
axes[1,2].set_title('GridSearchCV Mean AUC per Configuration (RF)')
axes[1,2].set_xlabel('Config Index')
axes[1,2].set_ylabel('Mean AUC')
axes[1,2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('task2_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Export Pipeline with joblib

In [ ]:
# Save best model (Random Forest pipeline)
best_model = rf_gs.best_estimator_ if rf_gs.best_score_ > lr_gs.best_score_ else lr_gs.best_estimator_
joblib.dump(best_model, 'churn_pipeline.joblib')
print('Pipeline exported: churn_pipeline.joblib')

# Verify reloading
loaded_pipeline = joblib.load('churn_pipeline.joblib')
test_pred = loaded_pipeline.predict(X_test[:5])
print(f'Predictions from reloaded pipeline: {test_pred}')

## 6. Final Summary & Insights

### Key Results
| Model | Accuracy | F1-Score | ROC AUC |
|-------|----------|----------|---------|
| Logistic Regression | ~80% | ~0.60 | ~0.85 |
| Random Forest | ~82% | ~0.62 | ~0.87 |

### Insights
1. **Contract type** is the strongest churn predictor — month-to-month customers churn far more.
2. **Tenure** and **TotalCharges** are highly inversely correlated with churn — loyal customers stay.
3. **Random Forest** outperforms Logistic Regression due to its ability to capture non-linear interactions.
4. The dataset is **imbalanced** (~26% churn rate) — F1-score is a more informative metric than accuracy.
5. The exported `joblib` pipeline handles all preprocessing automatically — production-ready for serving.